# Maximal visits, shifts and waiting times: demo

Concept: `overview/Single Level Visit Mappings.md` (outside this repo), sections "Visits", "Deriving maximal Visits", "Visit duration", "identifying shifts" and "Waiting Times"; waiting times as decided in Part C (with C.6, storage units) of the working notes (`claude_notes/excursions-and-visit-sequences.md`, 2026-10-31).
Code: modules `ocelescope_module_location_based_ocpm.visits.maximal`, `.shifts`, `.waiting`, `.pipeline` and `.visit_files`; this notebook only imports and calls their functions, so a change in the package shows up here on the next run.
Further detail: the docstrings (e.g. `help(transfers)`) and `docs/resources/visits.md`.

What the notebook does:
1. loads the running example (Level 3) and the small edge-case example, with their location context,
2. derives the maximal visits step by step,
3. finds the stops and transfers of every visit and categorises the transfers (shifts),
4. cuts every visit into waiting, storage and remaining time,
5. runs everything with one call (`analyse_visits`),
6. goes through the edge cases one by one on the small example,
7. saves the maximal visits of the running example to `data/running_example/derived/`.

Start Jupyter from the repository root with

```
uv run --with jupyter jupyter lab notebooks/visits.ipynb
```

Conventions in all frames: one row per hu-event relation $(e, hu)$ unless stated otherwise; `null` is $\bot$; durations are polars durations (e.g. `10h`).
All functions take and return polars frames; they work on eager (`DataFrame`) and lazy (`LazyFrame`) frames alike.

In [ ]:
from pathlib import Path

import polars as pl

from ocelescope_module_location_based_ocpm.derivation.level_files import MAPPING_SCHEMA, load_level_mapping
from ocelescope_module_location_based_ocpm.visits.context import add_high_level_locations, location_context
from ocelescope_module_location_based_ocpm.visits.maximal import (
    add_gaps,
    add_kappa,
    add_trace_positions,
    add_visit_ids,
    derive_maximal_visits,
    maximal_visits,
    visit_relations,
)
from ocelescope_module_location_based_ocpm.visits.pipeline import analyse_visits
from ocelescope_module_location_based_ocpm.visits.shifts import add_stops, shift_summary, transfers
from ocelescope_module_location_based_ocpm.visits.trace_figure import relations_for_figure, trace_figure
from ocelescope_module_location_based_ocpm.visits.visit_files import load_visits, save_visits
from ocelescope_module_location_based_ocpm.visits.waiting import (
    add_processing,
    add_storage_removals,
    time_pieces,
    waiting_summary,
    waiting_times,
)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
EXAMPLE = ROOT / "data" / "running_example"
DERIVED = EXAMPLE / "derived"
EDGE = ROOT / "data" / "visits_example"

# Show whole frames.
pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250)
pl.Config.set_fmt_table_cell_list_len(8)
pl.Config.set_fmt_str_lengths(40);

## 1. Inputs

### Running example

The location mapping of Level 3, as saved by `location_derivation.ipynb` (`load_level_mapping(folder, level)`), and the location context from `data/running_example/inputs/locations.csv` (`location_context(table)`, see `trace_figure.ipynb`, section 1): $agg$ maps A1, A2 → A and B1 → B; T is not mapped, so it is its own high-level location; $V = \{T\}$, storage units $W = \{B1\}$.

The excursion tolerance is $\kappa_{max} = 2$, as in the note's figure of the maximal visits.

In [ ]:
LEVEL, KAPPA = 3, 2
mapping = load_level_mapping(DERIVED, LEVEL)
context = location_context(pl.read_csv(EXAMPLE / "inputs" / "locations.csv"))
context

### Small edge-case example

`data/visits_example/` holds a small example made for this notebook (generated by `make_visits_example.py`, the single source of its files): seven handling units h1 to h7 whose traces cover the edge cases.
The relations are in the format of a saved level mapping (read with the same schema `MAPPING_SCHEMA`); `loc` is given directly, as if derived.

Location context: A1, A2, the **crane C1** (a vehicle mapped *into* A) and the **storage units S1, S2** lie in hall A, B1 and the storage unit B2 in hall B, and the **truck T** is not mapped (a vehicle with its own high-level location); $V = \{C1, T\}$, $W = \{S1, S2, B2\}$.
Processing operations $P$ are given explicitly in `processing_events.csv`; three of them are movements ($P \cap M \neq \emptyset$).
The example is meant for $\kappa_{max} = 1$.

| handling unit | edge cases (section 6) |
|---|---|
| h1 | all four transfer categories in one visit; vehicle shift via a vehicle inside (C1) and outside (T) of the high-level location; T visited twice (cut by κ); waiting time after an entry by processing |
| h2 | vehicle relations at the start and the end of a visit (not shifts); vehicle visits; a visit without processing |
| h3 | ⊥ relations (as at Level 0): two ⊥ visits because $\kappa(\bot) = 0$, ⊥ inside a transfer |
| h4 | a transferring visit (no stops, but a relation not on a vehicle); a high-level location visited twice (cut by κ) |
| h5 | waiting times: entry without processing, processing on a vehicle inside $hl$, processing during an excursion, processing movements as exit of one visit and entry of the next, trace end |
| h6 | storage units: a move between storage units (no removal), a removal, a processing operation that removes from storage (waits incl. the storage time, or 0 with `zero_wait_at_removal`), a removal outside A during an excursion |
| h7 | events with the same timestamp, ordered by AS4 against their ids: a removal (movement) before a processing operation, a processing operation before another event |

In [ ]:
EDGE_KAPPA = 1
edge_relations = pl.read_csv(EDGE / "relations.csv", schema=MAPPING_SCHEMA)
edge_context = location_context(pl.read_csv(EDGE / "locations.csv"))
edge_processing = pl.read_csv(EDGE / "processing_events.csv")["event"].to_list()
print(edge_context)
print("P =", edge_processing)
edge_relations

## 2. Maximal visits, step by step

Concept: a visit $(hu, hl, \sigma)$ is an infix $\sigma = otrace(hu)[i:k]$ that enters $hl$ at $e_i$, ends with the last event at $hl$ at $e_k$, and contains no deviating infix longer than $\kappa(hl)$ ($\kappa(\bot) = 0$).
The maximal visits are found as in the note's "Deriving maximal Visits": take the positions of $hu$ at $hl$, cut wherever the gap between two consecutive ones is longer than $\kappa(hl)$.
Each step below adds one or more columns to the relations frame; shown for c1, the note's example.

### Trace positions: `add_trace_positions(relations)`

Sorts the relations in $seq$ order per handling unit and adds `position`, the index $j$ of the relation in $otrace(hu) = \langle e_1, \dots, e_n \rangle$, starting at 1.
Events of one handling unit with the same timestamp are ordered by **AS4**: movement events first, then processing operations, then all other events, each group by event id (an event that is both a movement and a processing operation counts as a movement).
The group is the column `tie_rank` (0, 1, 2; `order.tie_rank`); the running example has no equal timestamps, h7 of the small example shows the effect (section 6.7).

| parameter | meaning | default |
|---|---|---|
| `relations` | one row per hu-event relation with columns hu, time, event, is_movement (and is_transformation if `processing` is `None`) | |
| `processing` | $P$ as event ids, for the tie-break; use the same $P$ as for the waiting times | `None` = $T$ |
| `movement` | column marking the movement events $M$ | `"is_movement"` |

In [ ]:
rel = add_trace_positions(mapping)
rel.filter(pl.col("hu") == "c1").select("hu", "event", "time", "is_movement", "is_transformation", "loc", "tie_rank", "position")

### High-level locations: `add_high_level_locations(relations, location, agg, high_level="high_level")`

$\widehat{hl}((e,hu)) = agg(loc((e,hu)))$ if $loc((e,hu)) \neq \bot$, else $\bot$; $agg$ may be partial and is completed by the identity (here: T ↦ T).

| parameter | meaning | default |
|---|---|---|
| `relations` | relations frame with the column `location` | |
| `location` | column of $loc$ | |
| `agg` | dict low-level → high-level location | identity |
| `high_level` | name of the new column | `"high_level"` |

In [ ]:
rel = add_high_level_locations(rel, "loc", context.agg)
rel.filter(pl.col("hu") == "c1").select("hu", "event", "loc", "position", "high_level")

### Tolerance: `add_kappa(relations, kappa, high_level="high_level")`

Adds `kappa` $= \kappa(\widehat{hl})$, the tolerance of the relation's own high-level location: $\kappa_{max}$ for every $hl \in HL$, 0 for ⊥.

| parameter | meaning | default |
|---|---|---|
| `relations` | relations frame with the column `high_level` | |
| `kappa` | $\kappa_{max} \geq 0$ | |
| `high_level` | column of $\widehat{hl}$ | `"high_level"` |

The concept has one $\kappa_{max}$; a tolerance per high-level location would only change this function.

In [ ]:
rel = add_kappa(rel, KAPPA)
rel.filter(pl.col("hu").is_in(["c1", "c5"])).select("hu", "event", "high_level", "kappa")

### Gaps: `add_gaps(relations, high_level="high_level", location="loc", vehicles=(), ignore_vehicles=False)`

Adds `gap` $g_j = p_j - p_{j-1} - 1$: the number of events since the previous relation of the same handling unit at the same high-level location (⊥ counts as one location here); null for the first one.

| parameter | meaning | default |
|---|---|---|
| `relations` | output of `add_trace_positions` with the column `high_level` | |
| `high_level` | column of $\widehat{hl}$ | `"high_level"` |
| `location`, `vehicles` | column of $loc$ and the set $V$; only used with `ignore_vehicles` | `"loc"`, empty |
| `ignore_vehicles` | `True`: relations on a vehicle ($loc \in V$) in between do not count towards the gap, even if the vehicle maps to another high-level location (see the end of this section); gaps before ⊥ relations always count every event | `False` |

How it works: every relation gets a weight $w$ (1, or 0 for a relation on a vehicle with `ignore_vehicles`); with $C$ the cumulative sum of $w$ per handling unit, the gap is $C(p_j) - w(p_j) - C(p_{j-1})$, which is $p_j - p_{j-1} - 1$ when all weights are 1. `over("hu", "high_level")` puts all ⊥ of a handling unit into one group, so ⊥ needs no stand-in value.

Example (note, table for κ = 2): c1 at A has positions 1, 2, 3, 6, 7 and gaps –, 0, 0, 2, 0.

In [ ]:
rel = add_gaps(rel)
rel.filter(pl.col("hu") == "c1").select("hu", "event", "position", "high_level", "gap")

### Visit ids: `add_visit_ids(relations, high_level="high_level")`

Adds `starts_visit` (true iff `gap` is null or `gap > kappa`: the relation starts a new maximal visit) and `visit_id`, the maximal visit at the relation's own high-level location.
The ids are numbered 1, 2, … over all visits, ordered by handling unit and first position.

How it works: a cumulative sum of `starts_visit` per (hu, high_level) numbers the visits; the visit's first position and the handling unit give the id (dense rank).

Example: with κ = 2, no gap of c1 is longer than 2, so c1 has one visit per high-level location (A, B, T).

In [ ]:
rel = add_visit_ids(rel)
rel.filter(pl.col("hu") == "c1").select("hu", "event", "position", "high_level", "gap", "kappa", "starts_visit", "visit_id")

### The visits: `maximal_visits(relations, high_level="high_level")`

One row per maximal visit $\nu = (hu, hl, otrace(hu)[i:k])$:

| column | meaning |
|---|---|
| `visit_id`, `hu`, `high_level`, `kappa` | the visit (high_level null = ⊥ visit) and its tolerance |
| `first_position`, `last_position` | $i$ and $k$ |
| `first_event`, `last_event` | $e_i$ (= the entry event $e_{in}$) and $e_k$ |
| `exit_event` | $e_{out} = otrace(hu)[k+1]$, or $e_k$ if the trace ends with the visit |
| `ends_trace` | true if the trace ends with the visit (then the duration is a lower bound) |
| `entry_time`, `last_time`, `exit_time` | times of $e_{in}$, $e_k$, $e_{out}$ |
| `duration` | $d = time(e_{out}) - time(e_{in})$ |
| `n_relations`, `n_at_location` | $\lvert\sigma\rvert$ and the number of relations at $hl$ |
| `interrupted` | true iff some relation of the visit is not at $hl$ |

How it works: group by `visit_id` (first/last position, event and time); the exit event is the relation of the same handling unit at `last_position + 1` (a join), if there is one.

These are exactly $m_1$ to $m_{11}$ of the note (κ = 2); the ids differ only in their order (the note starts with s1): visit 1 = $m_2$, 2 = $m_3$, 3 = $m_4$, 4 = $m_5$, 5 = $m_6$, 6 = $m_7$, 7 = $m_8$, 8 = $m_9$, 9 = $m_{10}$, 10 = $m_{11}$ (c5 at ⊥), 11 = $m_1$.

In [ ]:
visits = maximal_visits(rel)
visits

### Relations of the visits: `visit_relations(relations, visits, high_level="high_level")`

One row per visit and relation $r \in R(\nu)$, i.e. every relation of the visit's infix, including those of its excursions.
A relation therefore appears once for its own maximal visit and once for every visit at another location around it (e.g. e8 of c1 is in visit 3 = $m_4$ at T and, as an excursion, in visit 1 = $m_2$ at A).

| column added | meaning |
|---|---|
| `visit_id`, `visit_high_level`, `first_position`, `last_position` | the visit the row belongs to |
| `at_location` | $\widehat{hl}(r) = hl$ (for a ⊥ visit: $\widehat{hl}(r) = \bot$) |

How it works: each visit is expanded to its positions $i, \dots, k$ (`int_ranges` + `explode`) and joined with the relations on (hu, position); the cost is linear in the size of the result, no pairwise comparison of visits and relations.
It works for any set of visits given by hu, first and last position, not only maximal ones.

In [ ]:
vrel = visit_relations(rel, visits)
vrel.filter(pl.col("hu") == "c1").select("visit_id", "visit_high_level", "hu", "event", "position", "loc", "high_level", "at_location")

### All in one call: `derive_maximal_visits(relations, location="loc", agg=None, kappa=0, vehicles=(), ignore_vehicles=False, processing=None)`

Runs the steps above (trace positions, high-level locations, κ, gaps, visit ids, visits) and returns the relations and the visits.

| parameter | meaning | default |
|---|---|---|
| `relations` | e.g. a loaded level mapping or `Derivation.relations` | |
| `location` | column of $loc$ (e.g. `"loc_3"` on `Derivation.relations`) | `"loc"` |
| `agg` | location mapping | identity |
| `kappa` | $\kappa_{max}$ | 0 |
| `vehicles`, `ignore_vehicles` | see `add_gaps` | empty, `False` |
| `processing` | $P$, only for the order of events with the same timestamp (see `add_trace_positions`) | `None` = $T$ |

Below: the same visits again, and the number of maximal visits for κ = 0, 1, 2 (review 1: 13, 12, 10 at high-level locations, plus the ⊥ visit of c5).

In [ ]:
rel_k, visits_k = derive_maximal_visits(mapping, "loc", context.agg, KAPPA)
print("same as step by step:", visits_k.equals(visits))
{kappa: derive_maximal_visits(mapping, "loc", context.agg, kappa)[1].height for kappa in (0, 1, 2)}

### Option: vehicles do not use up the tolerance (`ignore_vehicles=True`)

By default, every event in a deviating infix counts towards $\kappa$, also relations on a vehicle that maps to another high-level location (here the truck T).
With `ignore_vehicles=True` (in `add_gaps`, `derive_maximal_visits` and `analyse_visits`; default `False`), relations on a vehicle do not count: a trip on the truck between two stays at A does not use up A's tolerance.
Visits to ⊥ are not affected (they allow no excursions at all).

Example with κ = 1: c1's relations between e5 and e9 are ⟨e7 at B1, e8 on T⟩. By default the gap is 2 > 1 and the visit to A is cut into e3–e5 and e9–e11; with `ignore_vehicles=True` only B1 counts, the gap is 1, and the visit to A stays e3–e11. T itself is still cut (e8 and e13–e14), since A2 is not a vehicle.

In [ ]:
for ignore in (False, True):
    rel_1, visits_1 = derive_maximal_visits(mapping, "loc", context.agg, 1, context.vehicles, ignore_vehicles=ignore)
    print(f"ignore_vehicles={ignore}: {visits_1.height} maximal visits")
    print(visits_1.filter(pl.col("hu") == "c1").select("visit_id", "high_level", "first_event", "last_event"))

The maximal visits in the trace figure (see `trace_figure.ipynb`): the figure only needs a `group` column next to the visit columns.

In [ ]:
HUS = ["s1", "c1", "c2", "c3", "c4", "c5"]
figure_relations = relations_for_figure(mapping, "loc", HUS, context.agg, context.vehicles, context.storage_units)
trace_figure(
    figure_relations,
    visits.with_columns(pl.lit("maximal").alias("group")),
    title=f"Running example, Level {LEVEL}: maximal visits at κ = {KAPPA}",
).show()

## 3. Stops, transfers and shifts

Concept ("identifying shifts"): the **stops** of a visit $\nu = (hu, hl, \sigma)$, $hl \in HL$, are its relations at $hl$ that are not on a vehicle.
Between two consecutive stops $a_j$ and $a_{j+1}$ lies the **transfer** $\mu_j = otrace(hu)[a_j + 1 : a_{j+1} - 1]$, possibly empty, with one category:

| category | condition |
|---|---|
| stay | $\mu_j = \langle\rangle$ and both stops have the same $loc$ |
| direct shift | $\mu_j = \langle\rangle$ and the stops have different $loc$ |
| vehicle shift | $\mu_j \neq \langle\rangle$ and all relations of $\mu_j$ are on vehicles |
| external shift | $\mu_j \neq \langle\rangle$ and some relation of $\mu_j$ is not on a vehicle (⊥ counts as not on a vehicle) |

Direct and vehicle shifts are internal shifts. Vehicle relations before the first and after the last stop are entry and exit movements, not shifts. Visits at ⊥ have no stops.

### Stops: `add_stops(visit_relations, vehicles=(), location="loc")`

Adds `on_vehicle` ($loc \in V$; false for ⊥) and `is_stop` (at the visit's $hl \in HL$ and not on a vehicle).

| parameter | meaning | default |
|---|---|---|
| `visit_relations` | output of `visit_relations` | |
| `vehicles` | the set $V$ | empty |
| `location` | column of $loc$ | `"loc"` |

Example (note, table for $m_5$): c2's visit to A (visit 4) has stops at positions 1, 2, 3, 5, 6; e8 at T is on a vehicle.

In [ ]:
vrel = add_stops(vrel, context.vehicles)
vrel.filter(pl.col("visit_id") == 4).select("visit_id", "event", "position", "loc", "high_level", "at_location", "on_vehicle", "is_stop")

### Transfers: `transfers(visit_relations, location="loc")`

One row per transfer $\mu_j$, $j = 1, \dots, m - 1$:

| column | meaning |
|---|---|
| `visit_id`, `hu`, `visit_high_level`, `j` | the visit and the number of the transfer |
| `from_position`, `to_position`, `from_event`, `to_event`, `from_loc`, `to_loc` | the stops $a_j$ and $a_{j+1}$ |
| `transfer_events` | the events of $\mu_j$ in trace order (empty list for $\langle\rangle$) |
| `transfer_length` | $\lvert\mu_j\rvert = a_{j+1} - a_j - 1$; 0 for adjacent stops (the reversed index range of the note) |
| `all_on_vehicle` | all relations of $\mu_j$ on vehicles (true for $\langle\rangle$) |
| `category` | stay, direct shift, vehicle shift, external shift |

How it works: the stops are numbered $j$ in trace order and paired with the next stop (`shift(-1)`); every other relation gets the number of stops before it (cumulative sum), so the relations of $\mu_j$ are the non-stops with number $j$; they are aggregated per (visit, $j$) and joined to the pairs.
Relations before the first stop (number 0) and after the last stop (number $m$) belong to no transfer.

Example (working doc A.2): c2's visit 4 = $m_5$: stay, direct shift (A1 → A2), vehicle shift via ⟨e8⟩ on T (although A2 → A2), stay; c1's visit 1 = $m_2$: stay, direct, external via ⟨e7 at B1, e8 at T⟩, stay.

In [ ]:
transfer_frame = transfers(vrel)
transfer_frame

### Per visit: `shift_summary(visit_relations, transfers_frame, location="loc")`

One row per visit:

| column | meaning |
|---|---|
| `n_stops` | $m$ |
| `n_stays`, `n_direct_shifts`, `n_vehicle_shifts`, `n_external_shifts` | transfers per category; they add up to $m - 1$ |
| `n_internal_shifts` | direct + vehicle shifts |
| `locations`, `locations_at_hl`, `stop_locations`, `non_vehicle_locations` | $L(\nu)$, $L_{hl}(\nu)$, $L_{stop}(\nu)$, $L_{\neg v}(\nu)$ (sorted lists, ⊥ left out) |
| `is_vehicle_visit` | $L_{\neg v}(\nu) = \emptyset$ |
| `is_transferring_visit` | no stops, but $L_{\neg v}(\nu) \neq \emptyset$ |

For visits at ⊥, the stop and shift columns and both properties are null (only defined for $hl \in HL$).

Example (note): $L(m_2) = \{A1, A2, B1, T\}$, $L_{stop}(m_2) = \{A1, A2\}$; vehicle visits $m_6$ and $m_8$ (visits 5 and 7); $m_4$ (visit 3, c1 at T) is a transferring visit.

In [ ]:
shifts = shift_summary(vrel, transfer_frame)
shifts

## 4. Waiting times, storage time and remaining time

Concept (working doc, Part C with C.6, decided 2026-10-31): a visit $\nu = (hu, hl, otrace(hu)[i:k])$, $hl \in HL$, is cut at its **clock events** and every piece is labelled by the event that ends it.
- **Processing positions** $Q(\nu)$: positions of the visit with $e_j \in P$ and $\widehat{hl}((e_j, hu)) = hl$ (processing at $hl$ only).
- **Storage removals** $X_W(\nu)$: positions $j \in \{i+1, \dots, k\}$ with $loc_{j-1} \in W$ and $loc_j \notin W$, anywhere (also outside $hl$, during an excursion). A move between two storage units is no removal. Option `outside_storage_resets=False` (in `add_storage_removals` and `analyse_visits`) counts only storage units inside $hl$ ($agg(loc_{j-1}) = hl$).
- Clock events: the entry $i$, $Q(\nu)$ and $X_W(\nu)$ in trace order, then the exit $e_{out}$.
- A piece ending at a processing operation is **waiting** time, one ending at a removal by another event **storage** time, the last one (ending at the exit) **remaining** time. So the waiting time before a processing operation counts from the latest of entry, previous processing operation and storage removal.
- A processing operation at the entry gets a waiting time of 0.
- A processing operation that itself takes the handling unit out of storage (concept since 2026-11-01): its waiting time counts from the previous clock event, so it **includes the time until and in storage**. Option `zero_wait_at_removal=True` (in `time_pieces` and `analyse_visits`) gives the earlier behaviour: the time before it is storage time and its waiting time is 0. Removals by other events reset the clock in both modes.
- waiting + storage + remaining $= d$; without storage units this is the earlier version (C.1).

$P$ is a separate input; if none is given, the transformation events $T$ are used. Processing operations may be movements; since $loc$ is the location after the event, a processing movement from $hl$ to $hl'$ counts at $hl'$.

### Processing operations: `add_processing(visit_relations, processing=None)`

Adds `is_processing_event` ($e \in P$, wherever it happens) and `is_processing` ($e \in P$ at the visit's $hl$; false at ⊥ visits).

| parameter | meaning | default |
|---|---|---|
| `visit_relations` | output of `visit_relations` (or `add_stops`) | |
| `processing` | $P$ as event ids | `None` = $T$ (column `is_transformation`) |

Running example with the default ($P = T = \{e1, e3, e15\}$): c1's visit 1 to A has one processing operation, the roll e3 (its entry).

In [ ]:
vrel = add_processing(vrel)
vrel.filter(pl.col("visit_id") == 1).select("visit_id", "event", "activity", "position", "loc", "high_level", "at_location", "is_processing_event", "is_processing")

### Storage removals: `add_storage_removals(visit_relations, storage_units=(), location="loc")`

Adds `is_storage_removal`: the relation is not the first of the visit, the previous relation of the handling unit is in a storage unit and this one is not ($loc_{j-1} \in W$, $loc_j \notin W$).

| parameter | meaning | default |
|---|---|---|
| `visit_relations` | output of `visit_relations` | |
| `storage_units` | the set $W$ | empty (no removals) |
| `location` | column of $loc$ | `"loc"` |
| `outside_storage_resets` | `True`: removals from any storage unit count (concept); `False`: only from storage units at the visit's $hl$ | `True` |

Running example, $W = \{B1\}$: c1 is put into storage at e7 (B1, during an excursion of its visit to A) and removed at e8 (onto the truck T); the removal also counts for the visit to A.

In [ ]:
vrel = add_storage_removals(vrel, context.storage_units)
vrel.filter(pl.col("visit_id") == 1).select("visit_id", "event", "position", "loc", "high_level", "is_processing", "is_storage_removal")

### The pieces: `time_pieces(visit_relations, visits)`

One row per piece: `visit_id`, `hu`, `visit_high_level`, `piece` (1, 2, … within the visit), `start_position`, `end_position`, `start_event`, `end_event`, `start_activity`, `end_activity`, `start_time`, `end_time`, `label` (waiting, storage, remaining) and `piece_duration`. Visits at ⊥ have none.

| parameter | meaning | default |
|---|---|---|
| `visit_relations` | output of `add_processing` and `add_storage_removals` | |
| `visits` | the visits (for the exit events) | |
| `activity` | column of the activity, looked up for the start and end event of each piece (left out if missing) | `"activity"` |
| `zero_wait_at_removal` | `True`: a processing operation that is itself a storage removal ends a storage piece and gets a waiting time of 0; `False`: its waiting time includes the storage time (concept) | `False` |

For a waiting piece, `end_activity` is the activity of the processing operation the waiting time belongs to, so waiting times can be grouped by activity directly (section 6.8).

How it works: the clock events of each visit (entry, processing, removals from `visit_relations`; the exit from `visits`) are sorted by position; each one after the entry ends a piece starting at the previous one (`shift(1)` per visit). A processing operation at the entry or at a removal gets an extra waiting piece of length 0.

Example: c1's visit 1: e3 (roll, entry) waiting 0, storage from e3 to the removal e8 (5 h), remaining from e8 to the exit e13 (5 h); $d = 10$ h.

In [ ]:
pieces = time_pieces(vrel, visits)
pieces

### Waiting times: `waiting_times(pieces)`

The pieces labelled "waiting", one per processing operation: `end_event` is the processing operation, `start_event` the clock event its waiting time starts at.

In [ ]:
waiting = waiting_times(pieces)
waiting

### Per visit: `waiting_summary(visits, pieces)`

One row per visit: `n_processing` ($\lvert Q(\nu) \rvert$), `n_storage_removals` (removals that end a storage piece), `total_waiting_time`, `total_storage_time` and `remaining_time`; they add up to the duration. All null for visits at ⊥.

In [ ]:
waiting_per_visit = waiting_summary(visits, pieces)
visits.select("visit_id", "hu", "high_level", "first_event", "exit_event", "duration").join(waiting_per_visit, on="visit_id")

The example of the working doc (C.6): with $P = \{e1, e3, e4, e15\}$ (cast slab, roll, label, cut), c1's visit to A ($m_2$) is cut into waiting 0 (e3), waiting 1 h (e4), storage 4 h (until the removal e8) and remaining 5 h (until e13): $0 + 1 + 4 + 5 = 10 = d$.

In [ ]:
pieces_c6 = time_pieces(add_processing(vrel, ["e1", "e3", "e4", "e15"]), visits)
pieces_c6.filter(pl.col("visit_id") == 1)

## 5. All in one call: `analyse_visits(relations, location="loc", agg=None, vehicles=(), kappa=0, processing=None, ignore_vehicles=False, storage_units=(), zero_wait_at_removal=False, outside_storage_resets=True)`

Runs sections 2 to 4 and joins the per-visit summaries into one visits table.

| parameter | meaning | default |
|---|---|---|
| `relations` | relations frame (eager or lazy) | |
| `location`, `agg`, `kappa` | as for `derive_maximal_visits` | `"loc"`, identity, 0 |
| `vehicles` | the set $V$ | empty |
| `processing` | $P$ as event ids; also orders events with the same timestamp (AS4, section 2) | `None` = $T$ |
| `ignore_vehicles` | relations on a vehicle do not count towards κ (section 2) | `False` |
| `storage_units` | the set $W$ (section 4) | empty |
| `zero_wait_at_removal` | processing at a storage removal waits 0 instead of including the storage time (section 4) | `False` |
| `outside_storage_resets` | removals from storage units outside $hl$ reset the clock (section 4) | `True` |

Returns a `VisitAnalysis` with the (collected) frames `relations`, `visits` (all per-visit columns), `visit_relations`, `transfers`, `pieces` and `waiting_times`.

In [ ]:
analysis = analyse_visits(mapping, "loc", context.agg, context.vehicles, kappa=KAPPA, storage_units=context.storage_units)
print("same visits as step by step:", analysis.visits.equals(visits.join(shifts, on="visit_id").join(waiting_per_visit, on="visit_id")))
analysis.visits

## 6. Edge cases on the small example

The small example with $\kappa_{max} = 1$ and the explicit $P$.
The figure uses the trace position axis, since the traces of h1 to h5 share no events.

In [ ]:
edge = analyse_visits(
    edge_relations, "loc", edge_context.agg, edge_context.vehicles, kappa=EDGE_KAPPA, processing=edge_processing,
    storage_units=edge_context.storage_units,
)
edge_figure_relations = relations_for_figure(
    edge_relations, "loc", ["h1", "h2", "h3", "h4", "h5", "h6", "h7"], edge_context.agg, edge_context.vehicles, edge_context.storage_units, axis="trace position"
)
trace_figure(
    edge_figure_relations,
    edge.visits.with_columns(pl.lit("maximal").alias("group")),
    title=f"Edge-case example: maximal visits at κ = {EDGE_KAPPA}",
).show()
edge.visits.drop("kappa", "first_position", "last_position", "entry_time", "last_time", "exit_time")

### 6.1 All four transfer categories, vehicles inside and outside $hl$ (h1)

h1 at A (e1–e10): A1, A1, A2, C1, A2, T, A1, B1, A2, A2.
- e1 → e2: adjacent, same place: **stay**;
- e2 → e3: adjacent, A1 → A2: **direct shift**;
- e3 → e5 via ⟨e4 on C1⟩: C1 is a vehicle *inside* A (so e4 is at A, but not a stop): **vehicle shift**, although A2 → A2 ("picked up and put back");
- e5 → e7 via ⟨e6 on T⟩: T is a vehicle *outside* A, so e6 uses up the tolerance (gap 1 ≤ κ = 1): **vehicle shift** as well; the category does not depend on whether a vehicle is mapped into $hl$;
- e7 → e9 via ⟨e8 at B1⟩: **external shift**;
- e9 → e10: **stay**.

In [ ]:
h1 = edge.visits.filter((pl.col("hu") == "h1") & (pl.col("high_level") == "A"))["visit_id"]
edge.transfers.filter(pl.col("visit_id").is_in(h1.implode())).drop("visit_high_level")

### 6.2 Vehicles at the start and end of a visit, vehicle visits (h2)

h2: T, C1, A1, A1, C1, T, T. The visit to A runs from e13 (picked up from the truck by the crane C1, which is in A) to e16 (picked up again by C1).
Its stops are e14 and e15 only; the crane relations before the first and after the last stop are entry and exit movements, so the visit has one stay and no shift.
Both visits to T contain only relations on vehicles: **vehicle visits**. The last one ends with the trace, so its duration (e17 to e18) is a lower bound (`ends_trace`).
The visit to A has no processing operation: its whole duration is remaining time.

In [ ]:
edge.visits.filter(pl.col("hu") == "h2").select(
    "visit_id", "high_level", "first_event", "last_event", "exit_event", "ends_trace", "duration",
    "n_stops", "n_stays", "n_internal_shifts", "locations", "non_vehicle_locations", "is_vehicle_visit",
    "n_processing", "remaining_time",
)

### 6.3 ⊥ (h3)

h3: ⊥, ⊥, A1, ⊥, A1, T, A2 (unknown locations in between, as at Level 0; from Level 1 on, ⊥ only occurs at the start of a trace).
- $\kappa(\bot) = 0$: e19–e20 is one ⊥ visit, e22 is a second one (gap 1 > 0); ⊥ visits have no stops, shifts or waiting times (null).
- The visit to A (e21–e25) contains e22 at ⊥: the transfer e21 → e23 via ⟨e22⟩ is an **external shift** (⊥ counts as not on a vehicle, "unknown" is the cautious choice); e23 → e25 via ⟨e24 on T⟩ is a vehicle shift.
- `locations` leaves ⊥ out.

In [ ]:
edge.visits.filter(pl.col("hu") == "h3").select(
    "visit_id", "high_level", "first_event", "last_event", "exit_event", "n_relations", "interrupted",
    "n_stops", "n_external_shifts", "n_vehicle_shifts", "locations", "is_vehicle_visit", "remaining_time",
)

In [ ]:
h3 = edge.visits.filter(pl.col("hu") == "h3")["visit_id"]
edge.transfers.filter(pl.col("visit_id").is_in(h3.implode())).drop("visit_high_level")

### 6.4 Transferring visit and cuts by κ (h4, h1)

h4: T, A1, T, T, B1, B1, A2.
- The visit to T (e26–e29) contains the excursion ⟨e27 at A1⟩ (gap 1 ≤ κ). T is a vehicle, so the visit has no stops, but A1 is not a vehicle: a **transferring visit** (not a vehicle visit).
- A is visited at e27 and at e32 with a gap of 4 > κ: **two maximal visits** to A. The same happens to h1 at T (e6 and e11).

In [ ]:
edge.visits.filter(pl.col("hu").is_in(["h4", "h1"]) & pl.col("high_level").is_in(["T", "A"])).select(
    "visit_id", "hu", "high_level", "first_event", "last_event", "n_relations", "n_stops",
    "locations", "non_vehicle_locations", "is_vehicle_visit", "is_transferring_visit",
)

With `ignore_vehicles=True` and κ = 0, h1's visit to A runs on across ⟨e6 on T⟩ (the truck trip uses no tolerance) and is only cut at e8 (B1); by default it is cut at e6 as well.

In [ ]:
for ignore in (False, True):
    _, visits_0 = derive_maximal_visits(edge_relations, "loc", edge_context.agg, 0, edge_context.vehicles, ignore_vehicles=ignore)
    print(f"ignore_vehicles={ignore}")
    print(visits_0.filter((pl.col("hu") == "h1") & (pl.col("high_level") == "A")).select("visit_id", "first_event", "last_event"))

### 6.5 Waiting times (h5, h1)

h5: A1 (crane move, entry), A1 (inspect), A1 (**roll** ∈ P), C1 (**weigh on crane** ∈ P), B1 (**paint and drop** ∈ P ∩ M), A2, A2 (label), B1 (**pack and move** ∈ P ∩ M), B1.
- Visit to A (e33–e39): the entry e33 is no processing operation, so $w_1 = 3 - 0 = 3$ h (roll), $w_2 = 4 - 3 = 1$ h (weighing **on the crane C1**, which is in A, so it counts).
- The paint e37 lies in an excursion of the A visit (at B), so it does **not** count for A; it is the entry of its own visit to B (e37) with $w_1 = 0$.
- The processing movement e40 ends at B1, so it counts at B: it is the **exit event** of the A visit ($w_{rem} = 10 - 4 = 6$ h, $d = 10 = 3 + 1 + 6$) and the **entry** of the next visit to B ($w_1 = 0$); that visit ends with the trace, so its remaining time is a lower bound.
- h1 at A: entry by processing (produce e1, $w_1 = 0$), then the grind e10 after 12 h ($w_2 = 12$ h), $w_{rem} = 1$ h.

In [ ]:
edge.pieces.filter(pl.col("hu").is_in(["h5", "h1"])).drop("visit_high_level", "start_time", "end_time")

In [ ]:
edge.visits.filter(pl.col("hu") == "h5").select(
    "visit_id", "high_level", "first_event", "exit_event", "ends_trace", "duration",
    "n_processing", "total_waiting_time", "remaining_time",
)

With the default $P = T$ (here only produce e1 and roll e35), h5's A visit has a single waiting time of 3 h and a remaining time of 7 h.

In [ ]:
edge_default = analyse_visits(edge_relations, "loc", edge_context.agg, edge_context.vehicles, kappa=EDGE_KAPPA)
edge_default.pieces.filter(pl.col("hu") == "h5").drop("visit_high_level", "start_time", "end_time")

### 6.6 Storage units (h6)

h6 at A: S1 (entry), S2, C1, A1, A1 (**roll** ∈ P), S1, A2 (**unpack and move** ∈ P ∩ M, takes it out of S1), B2 (storage in B, an excursion), A2, A2 (**grind** ∈ P), then the truck (exit).
- e43 moves it from S1 to S2: **no removal** (still in storage), so the clock keeps running;
- e44 takes it out of S2 onto the crane: **storage** piece of 5 h (e42 to e44); the roll e46 then waits 3 h (from the removal, not from the entry);
- e48 is a processing operation that itself removes from S1: its **waiting time is 4 h** (e46 to e48), including the time in S1;
- e50 takes it out of B2, a storage unit **outside A** (during the excursion): it still restarts the clock (Nina, 2026-10-31), so the grind e51 waits 1 h, not 3 h;
- remaining 2 h until the exit e52: $5 + 3 + 4 + 3 + 1 + 2 = 18$ h $= d$.

With `zero_wait_at_removal=True` (the second table), the 4 h before e48 are storage time instead and e48 gets a waiting time of 0: $5 + 3 + 4 + 0 + 3 + 1 + 2 = 18$ h.
With `outside_storage_resets=False` (the third table), the removal e50 from B2 (in B, outside A) no longer resets the clock, so the grind e51 waits 4 h from e48: $5 + 3 + 4 + 4 + 2 = 18$ h.

In the trace figure above, storage units are drawn as diamonds.

In [ ]:
edge.pieces.filter(pl.col("hu") == "h6").drop("visit_high_level", "start_time", "end_time")

In [ ]:
edge_zero = analyse_visits(
    edge_relations, "loc", edge_context.agg, edge_context.vehicles, kappa=EDGE_KAPPA, processing=edge_processing,
    storage_units=edge_context.storage_units, zero_wait_at_removal=True,
)
edge_zero.pieces.filter(pl.col("hu") == "h6").drop("visit_high_level", "start_time", "end_time")

In [ ]:
edge_inside = analyse_visits(
    edge_relations, "loc", edge_context.agg, edge_context.vehicles, kappa=EDGE_KAPPA, processing=edge_processing,
    storage_units=edge_context.storage_units, outside_storage_resets=False,
)
edge_inside.pieces.filter(pl.col("hu") == "h6").drop("visit_high_level", "start_time", "end_time")

In [ ]:
edge.visits.filter(pl.col("hu") == "h6").select(
    "visit_id", "high_level", "first_event", "exit_event", "duration",
    "n_processing", "n_storage_removals", "total_waiting_time", "total_storage_time", "remaining_time",
)

### 6.7 Events with the same timestamp (h7)

AS4 orders events with the same timestamp: movements, then processing operations, then the rest, each group by event id.
The ids of h7 are chosen against this order, so the event id order and AS4 differ at both ties:
- t = 3: the crane move **e55** takes h7 out of the storage unit S1 and the **grind e54** (∈ P) happens at the same time. AS4 puts the movement first, so e55 is the removal (storage piece of 3 h from the entry e53) and the grind waits **0 h** from it. In event id order, the grind would come first, be itself the removal, and wait 3 h (incl. the storage time).
- t = 5: the **roll e57** (∈ P) comes before the label e56 (not in P), so the roll waits 2 h from the grind e54 and the label falls into the remaining time.
- $3 + 0 + 2 + 1 = 6$ h $= d$.

Also, with the AS4 order, the grind at t = 3 is at A1 like the move at the same time (the handling unit is no longer in S1), which matches RQ4 (all relations of a handling unit with the same timestamp have the same location).
The trace figure uses the same order on every axis (on the time axis, tied events are spread in AS4 order).

In [ ]:
edge.relations.filter(pl.col("hu") == "h7").select("event", "time", "activity", "is_movement", "loc", "tie_rank", "position")

In [ ]:
edge.pieces.filter(pl.col("hu") == "h7").drop("visit_high_level", "start_time", "end_time")

### 6.8 Waiting times per activity

Every waiting time carries the activity of its processing operation (`end_activity`) and of the clock event it starts at (`start_activity`), so they can be grouped by activity directly, e.g. number, mean and maximum waiting time per processing activity:

In [ ]:
edge.waiting_times.group_by("end_activity").agg(
    pl.len().alias("n"),
    pl.col("piece_duration").mean().alias("mean_waiting_time"),
    pl.col("piece_duration").max().alias("max_waiting_time"),
).sort("end_activity")

### 6.9 Checks

For every visit at a high-level location: waiting, storage and remaining time add up to the duration, and the transfers add up to the number of stops minus one.

In [ ]:
for name, result in (("running example", analysis), ("edge-case example", edge)):
    located = result.visits.filter(pl.col("high_level").is_not_null())
    pieces_total = located["total_waiting_time"] + located["total_storage_time"] + located["remaining_time"]
    times_ok = pieces_total.equals(located["duration"], check_names=False)
    with_stops = located.filter(pl.col("n_stops") > 0)
    n_transfers = with_stops.select(pl.sum_horizontal("n_stays", "n_direct_shifts", "n_vehicle_shifts", "n_external_shifts"))
    transfers_ok = n_transfers.to_series().equals(with_stops["n_stops"] - 1, check_names=False)
    print(f"{name}: waiting + storage + remaining = d: {times_ok}; transfers = stops - 1: {transfers_ok}")

## 7. Save the maximal visits: `save_visits(visits, folder, level, kappa)` and `load_visits(folder, level, kappa)`

Writes the core columns of the maximal visits (those of `maximal_visits` without the duration; CSV has no duration or list type) to `data/running_example/derived/maximal_visits_level_{k}_kappa_{κ}.csv`, where later notebooks load them from; `load_visits` reads them and recomputes the duration.

In [ ]:
path = save_visits(analysis.visits, DERIVED, LEVEL, KAPPA)
print(path.relative_to(ROOT))
load_visits(DERIVED, LEVEL, KAPPA)